# DistilBERT · Kiểm tra rò rỉ và phân tích lỗi
**Lương Hoàng Bảo Quân · Nhóm 5 · NLP**

Mở notebook để đọc ngay kết quả đã lưu. **Lượt chạy mới** sử dụng checkpoint seed 42 ở bước 201, sau một epoch trên toàn bộ 6.420 bài đăng. Bộ demo có 10 câu tiếng Anh do người dùng cung cấp, nhãn đối chiếu nằm trong CSV. Phần cuối có số liệu Kaggle gốc của nhóm để tra cứu riêng.

**Chạy trên Colab**  
1. Tải cả notebook và ZIP. Trong Colab chọn **File → Upload notebook**, mở file .ipynb.
2. Để chạy lại, bấm nút ▶ ở ô đầu và chọn file ZIP khi được hỏi. Dữ liệu và trọng số có sẵn trong ZIP.
3. Chạy các ô lần lượt từ trên xuống. Mặc định ô dự đoán nạp checkpoint đã đóng gói. CPU đủ để chạy demo. GPU giúp đánh giá cả validation và test nhanh hơn. Ô cuối có tùy chọn huấn luyện lại.
4. Nếu dùng Jupyter trên máy, giải nén ZIP, mở notebook trong thư mục có `src/` và `data/`, rồi chạy các ô.

In [1]:
from pathlib import Path
import sys, os, zipfile, subprocess, json
ROOT = Path.cwd()
if not (ROOT / 'data/external_10_sentences.csv').exists():
    from google.colab import files
    print('Chọn DistilBERT_Nhom5.zip')
    uploaded = files.upload()
    archive = next(Path(p) for p in uploaded if p.lower().endswith('.zip'))
    with zipfile.ZipFile(archive) as z:
        z.extractall('du_an_nlp')
    ROOT = Path('du_an_nlp/DistilBERT_Nhom5').resolve()
    del uploaded
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
from importlib.metadata import version, PackageNotFoundError
from packaging.requirements import Requirement
def installed(line):
    requirement = Requirement(line)
    try:
        return version(requirement.name) in requirement.specifier
    except PackageNotFoundError:
        return False
if not all(installed(line) for line in Path('requirements.txt').read_text().splitlines() if line):
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
print('Sẵn sàng. Đã tìm thấy bộ dữ liệu và mã nguồn.')

Sẵn sàng. Đã tìm thấy bộ dữ liệu và mã nguồn.


## 1. Dữ liệu nào dùng để học?
Train cập nhật trọng số. Với lần đánh giá này, giữ toàn bộ train, ưu tiên validation trước test khi hai tập chia sẻ cùng nội dung. Trong validation và test chỉ giữ một dòng cho mỗi văn bản chuẩn hóa. Chuẩn hóa chuyển về chữ thường, bỏ URL và dấu câu rồi gộp khoảng trắng. Mô hình vẫn nhận văn bản gốc. Bảng giao nhau đếm **văn bản phân biệt**; bảng số mẫu đếm **dòng CSV**.

In [2]:
import pandas as pd, torch
from IPython.display import display
from src.splits import prepare
from src.analysis import check_demo, infer, analyse, save_table, threshold_experiment
from src.predict import load_model
raw, clean, pairs, stats, removed = prepare(Path('data/raw'),Path('data/clean'))
demo = pd.read_csv('data/external_10_sentences.csv')
display(pairs, stats)
print('Số câu demo theo nhãn:',demo.label.value_counts().to_dict())

Số câu demo theo nhãn: {'real': 5, 'fake': 5}


,split_1,split_2,raw_unique,normalized_unique
0,train,val,1,77
1,train,test,0,71
2,val,test,0,24


,split,raw,kept,removed,fake,real
0,train,6420,6420,0,3060,3360
1,val,2140,2045,95,955,1090
2,test,2140,2039,101,949,1090


## 2. Mười câu demo có bị đưa vào train?
Kiểm tra trùng nguyên văn, trùng sau chuẩn hóa và tìm câu gần nhất bằng TF-IDF ký tự dài 3–5. Bộ từ vựng và IDF được học từ train. Điểm cosine đo độ giống câu chữ. Hai câu có thể cùng phát biểu một ý bằng những từ khác nhau.

In [3]:
near = check_demo(demo,raw)
display(near.groupby('split').agg(trung_nguyen_van=('raw_match','sum'),trung_chuan_hoa=('normalized_match','sum')))
display(near.loc[near.split.eq('train'),['id','nearest_id','cosine','nearest_text']].round(3))
print('Nội dung gần nhất của từng câu trong cả ba tập nằm ở ket_qua/demo_overlap.csv.')

Nội dung gần nhất của từng câu trong cả ba tập nằm ở ket_qua/demo_overlap.csv.


,trung_nguyen_van,trung_chuan_hoa
split,,
test,0,0
train,0,0
val,0,0


,id,nearest_id,cosine,nearest_text
0,1,4369,0.348,Observing physical distance reduces the risk o...
1,2,1770,0.266,Wearing a mask can help slow the spread of COV...
2,3,3270,0.295,#IndiaFightsCorona 86052 new confirmed cases h...
3,4,3846,0.232,Chlorine dioxide has already proved efficaciou...
4,5,3524,0.290,A #symptomatic lab-confirmed #COVID19 case is ...
5,6,2694,0.300,Joe Biden said President Trump told Americans ...
6,7,5551,0.234,???Four kids who took the coronavirus vaccine ...
7,8,4045,0.203,vitamin D supplements can prevent a COVID-19 i...
8,9,5507,0.197,Scientists in the United States have discovere...
9,10,4711,0.355,Microchip implant with COVID-19 vaccine


## 3. Nạp DistilBERT và đọc nhật ký huấn luyện
Cấu hình gốc có lịch learning rate 4 epoch. Phiên train bị ngắt sau bước 328. Gói này giữ checkpoint hoàn chỉnh gần nhất ở **bước 201, hết epoch 1**. Mẫu train được gom theo độ dài để tiết kiệm padding. Tokenizer cắt ở 128 token, batch size 8, tích lũy gradient 4 bước, learning rate tối đa 2×10⁻⁵. Mã cập nhật trọng số nằm trong `src/finetune.py`.

In [4]:
folder = Path('checkpoint_epoch1')
cfg = json.loads((folder/'run_config.json').read_text())
state = json.loads((folder/'trainer_state.json').read_text())
tokenizer, model = load_model(folder)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Seed:',cfg['seed'],'| Bước:',state['global_step'],'| Epoch:',round(state['epoch'],2),'| Thiết bị suy luận:',device)
display(pd.DataFrame(state['log_history']).dropna(axis=1,how='all').tail(5))

Seed: 42 | Bước: 201 | Epoch: 1.0 | Thiết bị suy luận: cpu


,epoch,grad_norm,learning_rate,loss,step,eval_accuracy,eval_f1_fake,eval_f1_macro,eval_f1_weighted,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second
16,0.846824,7.928854,0.000018,0.2100,170,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,0.896638,5.422504,0.000017,0.1760,180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,0.946451,2.513284,0.000017,0.1549,190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,0.996264,0.171754,0.000017,0.0289,200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,1.000000,NaN,NaN,NaN,201,0.930073,0.923407,0.92954,0.929944,0.188784,20.5775,99.381,6.22


## 4. Chạy 10 câu demo và lấy từng mẫu sai
Mỗi hàng đi qua tokenizer và DistilBERT. Softmax cho điểm của hai lớp. Với ngưỡng mặc định, lớp có điểm cao hơn được chọn. Cuối cùng mới so với nhãn của file. Cột `confidence` đo mức tự tin của bộ phân loại.

In [5]:
frames = {}
out = Path('ket_qua_chay_lai')
def evaluate(name, frame):
    pred = infer(frame,tokenizer,model,cfg['max_length'],device)
    frames[name] = pred
    info,counts,errors = analyse(pred,out/name)
    print(f'{name}: đúng {len(pred)-len(errors)}/{len(pred)}, sai {len(errors)}, macro-F1 {info["f1_macro"]:.4f}')
    return counts,errors
demo_counts,demo_errors = evaluate('demo',demo.rename(columns={'sentence':'tweet'}))
cols = ['id','tweet','label','predicted_label','confidence','group']
with pd.option_context('display.max_colwidth',150):
    display(frames['demo'][cols].round(4))
display(demo_counts.query('total > 0'))
print('ID đoán sai:',demo_errors.id.sort_values().tolist())

demo: đúng 5/10, sai 5, macro-F1 0.3333
ID đoán sai: [1, 2, 3, 4, 5]


,id,tweet,label,predicted_label,confidence,group
0,1,WHO recommends washing your hands frequently with soap and water to reduce the risk of infection.,real,fake,0.9843,Khuyến cáo phòng dịch
1,2,Wearing a mask in crowded indoor places can help reduce the spread of COVID-19.,real,fake,0.9946,Khuyến cáo phòng dịch
2,3,"The health ministry reported 1,250 new confirmed COVID-19 cases and 14 deaths in the last 24 hours.",real,fake,0.9918,Số liệu dịch tễ
3,4,Vaccines are approved only after clinical trials show they are safe and effective.,real,fake,0.9903,Vắc-xin / nghiên cứu
4,5,"People with symptoms such as fever, cough or loss of taste or smell should get tested and isolate.",real,fake,0.9666,Khuyến cáo phòng dịch
5,6,Drinking bleach or disinfectant kills the coronavirus inside your body.,fake,fake,0.9939,Mẹo chữa / phòng bệnh
6,7,"5G mobile towers spread the coronavirus, so they must be shut down immediately.",fake,fake,0.9947,Âm mưu / công nghệ
7,8,Eating raw garlic every morning completely prevents COVID-19 infection.,fake,fake,0.9943,Mẹo chữa / phòng bệnh
8,9,Breaking: the government is hiding a secret cure for coronavirus that works in 24 hours.,fake,fake,0.9917,Âm mưu / công nghệ
9,10,COVID-19 vaccines contain microchips that let the government track everyone.,fake,fake,0.9946,Âm mưu / công nghệ


,group,total,errors,fake_to_real,real_to_fake,error_rate
0,Âm mưu / công nghệ,3,0,0,0,0.0
2,Mẹo chữa / phòng bệnh,2,0,0,0,0.0
3,Số liệu dịch tễ,1,1,0,1,1.0
4,Vắc-xin / nghiên cứu,1,1,0,1,1.0
5,Khuyến cáo phòng dịch,3,3,0,3,1.0


**Đọc 5 lỗi demo.** Mô hình gán cả 10 câu là fake. Ba lỗi thuộc nhóm khuyến cáo (1, 2, 5), một lỗi thống kê (3) và một lỗi vắc-xin (4). Câu 3 có số liệu nhưng thiếu địa điểm, ngày và nguồn cụ thể; nhãn real ở đây là nhãn người soạn cung cấp. Với bộ 10 câu, mỗi câu tương ứng 10 điểm phần trăm accuracy. Kết quả này chưa đủ để kết luận về mọi bài đăng bên ngoài.

## 5. Phân tích lỗi trên validation và đánh giá test
Phân nhóm theo từ khóa **trước khi xem nhãn dự đoán**. Một câu thuộc nhóm khớp đầu tiên trong `src/analysis.py`. Nhóm “Khác” giữ các câu cần đọc tay. Trong bảng, `total` là tổng mẫu, `errors` là số sai, `error_rate` là số sai chia tổng mẫu trong nhóm. Nhóm không có mẫu để trống tỷ lệ. Hai cột chiều nhầm cộng lại bằng số lỗi.

In [6]:
val_counts,val_errors = evaluate('val',clean['val'])
display(val_counts.round(4))
with pd.option_context('display.max_colwidth',160):
    display(val_errors.groupby('group',sort=False).head(1)[cols+['matched_text','hypothesis','how_to_check']])
print('Tất cả mẫu sai có trong ket_qua/val/mau_sai.csv.')

val: đúng 1902/2045, sai 143, macro-F1 0.9295
Tất cả mẫu sai có trong ket_qua/val/mau_sai.csv.


,group,total,errors,fake_to_real,real_to_fake,error_rate
0,Âm mưu / công nghệ,22,3,3,0,0.1364
1,Đính chính / phủ định,36,11,10,1,0.3056
2,Mẹo chữa / phòng bệnh,61,4,4,0,0.0656
3,Số liệu dịch tễ,491,24,15,9,0.0489
4,Vắc-xin / nghiên cứu,89,7,5,2,0.0787
5,Khuyến cáo phòng dịch,130,13,4,9,0.1000
6,Chính sách / sự kiện,192,20,9,11,0.1042
7,Khác / cần đọc tay,1024,61,43,18,0.0596


,id,tweet,label,predicted_label,confidence,group,matched_text,hypothesis,how_to_check
1499,1567,There is no one in New Zealand receiving hospital-level care for COVID-19.,real,fake,0.994086,Khác / cần đọc tay,,Quy tắc chưa chỉ ra cơ chế cụ thể.,"Đọc toàn câu, kiểm tra nhãn, nguồn, trích dẫn và ngữ cảnh."
786,814,A better milestone. New York City saw at least one COVID-19 death on every day since mid March. That's 120+ days. Yesterday after months of communal effort ...,real,fake,0.990492,Số liệu dịch tễ,deaths,Văn phong bản tin và con số có thể ảnh hưởng dự đoán dù câu thiếu bối cảnh.,"Đối chiếu cùng phát biểu khi giữ hoặc ẩn con số, ngày và nguồn."
1020,1064,Hollywood actor Paul Rudd has encouraged young people to wear face masks in an entertaining video as the US grapples with the #COVID19 pandemic https://t.co...,real,fake,0.984827,Khuyến cáo phòng dịch,masks,Lời khuyên ngắn có thể thiếu các dấu hiệu nguồn thường thấy trong dữ liệu train.,So cùng một khuyến cáo khi có và khi bỏ cụm dẫn nguồn.
650,671,#FactCheck: Is #mouthwash proving to be effective against #Coronavirus? Researchers surely have found encouraging results. NOTE: Mouthwash is not a cure but...,fake,real,0.982000,Mẹo chữa / phòng bệnh,cure,"Tên chất hoặc cách chữa có thể che mất khác biệt giữa mô tả, bác bỏ và khẳng định hiệu quả.","Giữ nguyên chất được nhắc đến, thay cách khẳng định bằng câu bác bỏ có nguồn."
1541,1614,@thereidout @JoyAnnReid Airborne #COVID19 is the pre-existing condition that’ll financially bury American #families whose states’ governors are manipulating...,fake,real,0.981925,Chính sách / sự kiện,school,Tên cơ quan và sự kiện có thể trở thành dấu hiệu thay cho nội dung cần phân loại.,So hai tin cùng sự kiện nhưng khác nguồn hoặc lập trường.
49,50,A coronavirus vaccine: Where does it stand? Experts we checked with remain optimistic about a vaccine in the first quarter of 2021. https://t.co/XAwbq9IqrH ...,fake,real,0.979468,Vắc-xin / nghiên cứu,vaccine,Câu mô tả nghiên cứu và câu đưa ra tuyên bố chưa kiểm chứng có thể dùng chung từ chuyên môn.,So các câu cùng chủ đề có và không nêu rõ đối tượng hoặc kết quả thử nghiệm.
1273,1330,Fact Check's #CoronaCheck newsletter appeared on @ABCmediawatch in aid of their takedown of bunkum 5G claims in the media. It's your inoculation against mis...,fake,real,0.930178,Âm mưu / công nghệ,5G,Mô hình có thể dựa vào từ khóa công nghệ mà bỏ sót người đang khẳng định hay bác bỏ.,So hai câu cùng nhắc một tin đồn nhưng khác lập trường.
569,590,A cohort of nearly 50 fact checking organisations from @factchecknet have been working in the past week to debunk the misinformation emerging about the #cor...,fake,real,0.911444,Đính chính / phủ định,fact checking,Phần nhắc lại tin đồn có thể nổi bật hơn phần đính chính.,"Đánh dấu ai nói gì, từ phủ định nằm trong hay ngoài lời trích."


**Một giả thuyết có thể kiểm tra.** Nhóm đính chính/phủ định có 11/36 lỗi validation, trong đó 10 câu mang nhãn fake bị đoán real. Có thể mô hình xem cách dẫn nguồn hoặc từ “fact check” là dấu hiệu tin thật. Với nhóm này cần đọc cả mệnh đề đang được trích và đối chiếu quy tắc gán nhãn. Riêng demo, cả năm câu real đều sai dù có câu nhắc WHO. Chỉ riêng việc nêu tên cơ quan chưa giải thích được kết quả.

In [7]:
train = raw['train'].assign(has_url=raw['train'].tweet.str.contains(r'https?://',regex=True))
display(pd.crosstab(train.label,train.has_url,normalize='index').round(3))
print('Tỷ lệ URL theo nhãn trong train là dấu hiệu để thiết kế phép thử về văn phong và nguồn.')

Tỷ lệ URL theo nhãn trong train là dấu hiệu để thiết kế phép thử về văn phong và nguồn.


has_url,False,True
label,,
fake,0.673,0.327
real,0.308,0.692


## 6. Thử cải tiến và đo lại
**A/B đã chạy.** A dùng ngưỡng fake 0,50. B giữ nguyên model, chọn ngưỡng tối đa hóa macro-F1 trên 1.022 mẫu validation. So A và B trên 1.023 mẫu validation còn lại với cùng ID. Xét cả hai chiều nhầm, vì giảm gắn nhầm tin thật có thể đồng thời làm bỏ lọt tin giả.

In [8]:
ab = threshold_experiment(frames['val'],out/'thu_nghiem')
display(ab.round(4))
print('B trừ A về số lỗi:',int(ab.errors.iloc[1]-ab.errors.iloc[0]))

B trừ A về số lỗi: 4


,variant,threshold,n_dev,n_check,errors,macro_f1,accuracy,fake_to_real,real_to_fake
0,A,0.50,1022,1023,60,0.9409,0.9413,39,21
1,B,0.63,1022,1023,64,0.9369,0.9374,46,18


Trong lần chạy đã lưu, B dùng ngưỡng 0,63. Số real bị gắn fake giảm từ 21 xuống 18, nhưng số fake bị bỏ lọt tăng từ 39 lên 46. Tổng lỗi tăng từ 60 lên 64. **Giữ ngưỡng 0,50** cho đánh giá cuối.

**Thí nghiệm tiếp theo cho nhóm phủ định.** Lấy bài từ nguồn độc lập với demo, validation và test. Hai người đọc gán nhãn các cặp cùng chủ đề nhưng trái lập trường, rà lại trường hợp bất đồng và loại bản sao. A dùng train hiện tại. B thay một phần bài train bằng các cặp mới, giữ số mẫu, số bước cập nhật, tỷ lệ lớp và cấu hình. Chạy cùng các seed 42, 123, 2024. So macro-F1 và tỷ lệ sai của nhóm phủ định trên cùng validation. Kiểm tra lại bằng một tập ngoài mới sau khi chốt phương án. Đây là thiết kế cho lượt sau.

In [9]:
test_counts,test_errors = evaluate('test',clean['test'])
display(pd.crosstab(frames['test'].label,frames['test'].predicted_label,rownames=['Nhãn gốc'],colnames=['Dự đoán']))
display(test_counts.round(4))

test: đúng 1896/2039, sai 143, macro-F1 0.9294


Dự đoán,fake,real
Nhãn gốc,,
fake,868,81
real,62,1028


,group,total,errors,fake_to_real,real_to_fake,error_rate
0,Âm mưu / công nghệ,20,1,1,0,0.0500
1,Đính chính / phủ định,31,5,5,0,0.1613
2,Mẹo chữa / phòng bệnh,61,1,1,0,0.0164
3,Số liệu dịch tễ,440,34,12,22,0.0773
4,Vắc-xin / nghiên cứu,102,10,7,3,0.0980
5,Khuyến cáo phòng dịch,170,15,7,8,0.0882
6,Chính sách / sự kiện,172,8,4,4,0.0465
7,Khác / cần đọc tay,1043,69,44,25,0.0662


## 7. Đối chiếu số liệu nhóm đã nộp
Bảng sau đọc báo cáo gốc trên nhánh `quan/distilbert-results`. Số liệu ở đó dùng 2.140 mẫu test gốc và các checkpoint khác. Lượt mới ở trên dùng test đã lọc 2.039 mẫu. Hai bảng trả lời cho hai điều kiện đánh giá khác nhau.

In [10]:
rows = []
for seed in (42,123,2024):
    base = Path('doi_chieu_github')/f'seed{seed}'
    ref = json.loads((base/'metrics.json').read_text())
    test_info = json.loads((base/'test_metrics.json').read_text())
    rows.append(dict(seed=seed,n_train=ref['n_train'],n_val=ref['n_val'],
                     val_macro_f1=ref['val']['f1_macro'],test_macro_f1=test_info['f1_macro']))
display(pd.DataFrame(rows).round(4))

,seed,n_train,n_val,val_macro_f1,test_macro_f1
0,42,6420,2140,0.9719,0.9695
1,123,6420,2140,0.9752,0.9728
2,2024,6420,2140,0.9714,0.9724


## 8. Tùy chọn huấn luyện lại từ mô hình nền
Để tái huấn luyện đủ 4 epoch, bật GPU, đổi `TRAIN_LAI` thành `True` và chạy ô dưới. Cần mạng cho lần tải mô hình nền; dữ liệu đã có sẵn trong ZIP. Đường dẫn mới tránh ghi đè checkpoint đóng gói. Mô hình mới xuất ở `lan_train_moi/.../model/`. Để đánh giá nó, nạp model từ thư mục đó rồi chạy lại các mục 4–6. Số liệu trình bày sẵn trong Markdown thuộc bản epoch 1.

In [11]:
TRAIN_LAI = False
if TRAIN_LAI:
    from datetime import datetime
    run = Path('lan_train_moi')/datetime.now().strftime('%Y%m%d_%H%M%S')
    subprocess.run([sys.executable,'-m','src.finetune','--seed','42','--config','configs/distilbert.json',
                    '--data-dir','data/clean','--out-dir',str(run)],check=True)
    tokenizer,model = load_model(run/'model')
    print('Đã nạp model vừa huấn luyện:',run)
else:
    print('Đang dùng checkpoint đóng gói. Kết quả tải sẵn ở ket_qua/. Kết quả chạy lại ở ket_qua_chay_lai/.')

Đang dùng checkpoint đóng gói. Kết quả tải sẵn ở ket_qua/. Kết quả chạy lại ở ket_qua_chay_lai/.
